In [1]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
from collections import Counter
import seaborn as sns
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import warnings
import copy
from tqdm import tqdm
from joblib import dump, load

In [2]:
warnings.filterwarnings('ignore')

Аналитическое решение задачи линейной регрессии в векторной форме:

$$
\mathbf{w} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}
$$

где:

$$
\mathbf{w} \quad\text{— вектор параметров модели (веса),}
$$

$$
\mathbf{X} \quad\text{— матрица признаков (размерность } n \times d, \text{ где } n \text{ — число объектов, } d \text{ — число признаков),}
$$

$$
\mathbf{y} \quad\text{— вектор целевых значений (размерность } n \times 1).
$$

Это решение минимизирует функцию потерь среднеквадратичной ошибки:

$$
J(\mathbf{w}) = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2_2
$$


Аналитическое решение задачи линейной регрессии с L2-регуляризацией (ригрессия Риджа):

$$
\mathbf{w} = (\mathbf{X}^\top \mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^\top \mathbf{y}
$$

где:

$$
\lambda > 0 \quad\text{— коэффициент регуляризации,}
$$

$$
\mathbf{I} \quad\text{— единичная матрица размера } d \times d.
$$

Это решение минимизирует функцию потерь с регуляризацией:

$$
J(\mathbf{w}) = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2_2 + \lambda \|\mathbf{w}\|^2_2
$$


Задача линейной регрессии с L1-регуляризацией (Lasso) минимизирует функцию потерь:

$$
J(\mathbf{w}) = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2_2 + \lambda \|\mathbf{w}\|_1
$$

где:

$$
\lambda > 0 \quad\text{— коэффициент регуляризации,}
$$

$$
\|\mathbf{w}\|_1 = \sum_{j=1}^d |w_j| \quad\text{— L1-норма вектора весов.}
$$

L1 штрафует сумму абсолютных значений весов, если вес оказывается ранвым 0, значит этот признак не влияет на модель.

Можно сделать модель нелинейной по признакам (PolynomialFeatures и Kernel Ridge)

In [3]:
df_train = pd.read_json('data/train.json')
df_test = pd.read_json('data/test.json')

df_train = df_train[(df_train['price'] >= 500) & (df_train['price'] <= 20000)]
df_test = df_test[(df_test['price'] >= 500) & (df_test['price'] <= 20000)]

df_train = df_train[(df_train['latitude'] > 40) & (df_train['latitude'] < 41.5)]
df_test = df_test[(df_test['longitude'] > -75) & (df_test['longitude'] < -72)]
print(df_train.columns)

Index(['bathrooms', 'bedrooms', 'building_id', 'created', 'description',
       'display_address', 'features', 'latitude', 'listing_id', 'longitude',
       'manager_id', 'photos', 'price', 'street_address', 'interest_level'],
      dtype='object')


In [4]:
mapping = {'low': 0, 'medium': 1, 'high': 2}
df_train['interest_level'] = df_train['interest_level'].map(mapping)

In [5]:
all_features = []

for _, row in df_train.iterrows():
    all_features.extend(row['features'])

In [6]:
counter = Counter(all_features)

counter_no_spaces = Counter()
for feature, count in counter.items():
    counter_no_spaces[feature.replace(" ", "")] += count

print(f'Всего уникальных признаков: {len(counter_no_spaces)}')

top_20_features = counter_no_spaces.most_common(20)
#print(f'Топ-20 уникальных признаков: {top_20_features}')
top_20_features

Всего уникальных признаков: 1538


[('Elevator', 25819),
 ('HardwoodFloors', 23470),
 ('CatsAllowed', 23464),
 ('DogsAllowed', 21963),
 ('Doorman', 20802),
 ('Dishwasher', 20369),
 ('NoFee', 18014),
 ('LaundryinBuilding', 16298),
 ('FitnessCenter', 13200),
 ('Pre-War', 9125),
 ('LaundryinUnit', 8679),
 ('RoofDeck', 6520),
 ('OutdoorSpace', 5235),
 ('DiningRoom', 5092),
 ('HighSpeedInternet', 4283),
 ('Balcony', 2965),
 ('SwimmingPool', 2715),
 ('LaundryInBuilding', 2586),
 ('NewConstruction', 2557),
 ('Terrace', 2251)]

In [7]:
top_20_features_keys = [k for k, _ in top_20_features]

for feature in top_20_features_keys:
    df_train[feature] = df_train['features'].apply(lambda x: int(feature in [f.replace(" ", "") for f in x]))

for feature in top_20_features_keys:
    df_test[feature] = df_test['features'].apply(lambda x: int(feature in [f.replace(" ", "") for f in x]))

In [8]:
feature_list = []

for feature in top_20_features_keys:
    feature_list.append(feature)

feature_list.append('bathrooms')
feature_list.append('bedrooms')

теперь становится 22 фичи

In [9]:
np.random.seed(21)

class LinearRegressionSGD:
    def __init__(self, learning_rate=0.01, n_iter=1000):
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.weights = None
        self.bias = None


    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in tqdm(range(self.n_iter), desc='Epochs'):
            for idx in range(n_samples):
                xi = X.iloc[idx].values
                yi = y.iloc[idx]

                y_pred = np.dot(xi, self.weights) + self.bias
                error = y_pred - yi

                self.weights -= self.learning_rate * error * xi
                self.bias -= self.learning_rate * error


    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

In [10]:
np.random.seed(21)

class LinearRegressionAnalytical:
    def __init__(self):
        self.weights = None
        self.bias = None


    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]

        theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

        self.bias = theta[0]
        self.weights = theta[1:]


    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

In [11]:
np.random.seed(21)

class LinearRegressionGD:
    def __init__(self, learning_rate=0.01, n_iter=1000):
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.weights = None
        self.bias = None


    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in tqdm(range(self.n_iter), desc='Epochs'):
            y_pred = np.dot(X, self.weights) + self.bias
            error = y_pred - y

            dw = (1 / n_samples) * np.dot(X.T, error)
            db = (1 / n_samples) * np.sum(error)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db


    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

In [12]:
model_analytical = LinearRegressionAnalytical()
model_SGD = LinearRegressionSGD()
model_GD = LinearRegressionGD()

In [13]:
X_train = df_train[feature_list]
y_train = df_train['price']
X_test = df_test[feature_list]
y_true = df_test['price'] 

In [14]:
from joblib import dump
import numpy as np
import time


def clean_model(model):
    for attr in ['X', 'y', 'history', 'grad_history', 'loss_history']:
        if hasattr(model, attr):
            setattr(model, attr, None)
    return model


def save_with_timer(model, filename):
    start = time.time()
    
    dump(model, filename, compress=0)
    
    end = time.time()
    print(f"{filename} saved in {end - start:.3f} sec")


model_analytical = clean_model(model_analytical)
model_SGD = clean_model(model_SGD)
model_GD = clean_model(model_GD)

save_with_timer(model_analytical, 'model_analytical.pkl')
save_with_timer(model_SGD, 'model_SGD.pkl')
save_with_timer(model_GD, 'model_GD.pkl')

model_analytical.pkl saved in 0.001 sec
model_SGD.pkl saved in 0.001 sec
model_GD.pkl saved in 0.000 sec


In [15]:
model_analytical = load('misc/model_analytical.pkl')
model_SGD = load('misc/model_SGD.pkl')
model_GD = load('misc/model_GD.pkl')

In [16]:
def r2_score(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    
    return 1 - (ss_res / ss_tot)

In [19]:
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])
result_R2 = pd.DataFrame(columns=['model', 'train', 'test'])

lr = LinearRegression()
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [20]:
def evaluate_model(model, X_train, X_test, name):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)
    
    mae_test = mean_absolute_error(y_true, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_true, y_pred_test))
    r2_test = r2_score(y_true, y_pred_test)

    mae_train = f'{mae_train:.4f}'
    rmse_train = f'{rmse_train:.4f}'
    r2_train = f'{r2_train:.4f}'

    mae_test = f'{mae_test:.4f}'
    rmse_test = f'{rmse_test:.4f}'
    r2_test = f'{r2_test:.4f}' 

    result_MAE.loc[len(result_MAE)] = [name, mae_train, mae_test]
    result_RMSE.loc[len(result_RMSE)] = [name, rmse_train, rmse_test]
    result_R2.loc[len(result_R2)] = [name, r2_train, r2_test]

In [21]:
evaluate_model(lr, X_train, X_test, 'Linear Regression')
evaluate_model(model_analytical, X_train, X_test, 'Linear Regression Analytical')
evaluate_model(model_SGD, X_train, X_test, 'Linear Regression SGD')
evaluate_model(model_GD, X_train, X_test, 'Linear Regression GD')

In [22]:
print(result_MAE)

                          model     train      test
0             Linear Regression  786.4448  791.6552
1  Linear Regression Analytical  786.4448  791.6552
2         Linear Regression SGD  795.9364  800.7231
3          Linear Regression GD  784.3144  788.4456


In [23]:
print(result_RMSE)

                          model      train       test
0             Linear Regression  1252.4692  1498.6112
1  Linear Regression Analytical  1252.4692  1498.6112
2         Linear Regression SGD  1299.8644  1507.6000
3          Linear Regression GD  1263.7868  1445.7239


In [24]:
print(result_R2)

                          model   train    test
0             Linear Regression  0.5736  0.3888
1  Linear Regression Analytical  0.5736  0.3888
2         Linear Regression SGD  0.5408  0.3814
3          Linear Regression GD  0.5659  0.4312


In [25]:
class LinearRegressionRegularized:
    def __init__(self, learning_rate=0.01, n_iter=10000, penalty=None, alpha=500, l1_ratio=0.5):
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.penalty = penalty
        self.alpha = alpha
        self.l1_ratio = l1_ratio

        self.weights = None
        self.bias = None


    def fit(self, X, y):
        X = X.to_numpy() if hasattr(X, 'to_numpy') else X
        y = y.to_numpy() if hasattr(y, 'to_numpy') else y
        y = y.flatten()

        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in tqdm(range(self.n_iter), desc="Training"):
            y_pred = np.dot(X, self.weights) + self.bias
            error = y_pred - y

            dw = (1 / n_samples) * np.dot(X.T, error)
            db = (1 / n_samples) * np.sum(error)

            if self.penalty == 'l2':  
                dw += (self.alpha / n_samples) * self.weights

            elif self.penalty == 'l1':  
                dw += (self.alpha / n_samples) * np.sign(self.weights)

            elif self.penalty == 'elasticnet':
                l1 = self.l1_ratio * np.sign(self.weights)
                l2 = (1 - self.l1_ratio) * self.weights
                dw += (self.alpha / n_samples) * (l1 + l2)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db


    def predict(self, X):
        X = X.to_numpy() if hasattr(X, 'to_numpy') else X
        return np.dot(X, self.weights) + self.bias

In [26]:
model_l1 = LinearRegressionRegularized(penalty='l1')
model_l2 = LinearRegressionRegularized(penalty='l2')
model_elasticnet = LinearRegressionRegularized(penalty='elasticnet')

In [27]:
model_l1.fit(X_train, y_train)
model_l2.fit(X_train, y_train)
model_elasticnet.fit(X_train, y_train)

Training: 100%|█████████████████████████████████████████████████████████| 10000/10000 [00:01<00:00, 7088.00it/s]


In [28]:
evaluate_model(model_l1, X_train, X_test, 'Linear Regression L1')
evaluate_model(model_l2, X_train, X_test, 'Linear Regression L2')
evaluate_model(model_elasticnet, X_train, X_test, 'ElasticNet custom')

In [29]:
ridge_reg = Ridge()
lasso_reg = Lasso()
elasticnet_reg = ElasticNet(l1_ratio=0.5)

In [30]:
ridge_reg.fit(X_train, y_train)
lasso_reg.fit(X_train, y_train)
elasticnet_reg.fit(X_train, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the penalty terms. Defaults to 1.0.See the notes for the exact mathematical meaning of thisparameter. ``alpha = 0`` is equivalent to an ordinary least square,solved by the :class:`LinearRegression` object. For numericalreasons, using ``alpha = 0`` with the ``Lasso`` object is not advised.Given this, you should use the :class:`LinearRegression` object.",1.0
,"l1_ratio l1_ratio: float, default=0.5The ElasticNet mixing parameter, with ``0 <= l1_ratio <= 1``. For``l1_ratio = 0`` the penalty is an L2 penalty. ``For l1_ratio = 1`` itis an L1 penalty. For ``0 < l1_ratio < 1``, the penalty is acombination of L1 and L2.",0.5
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If ``False``, thedata is assumed to be already centered.",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.Check :ref:`an example on how to use a precomputed Gram Matrix in ElasticNet`for details.",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [31]:
evaluate_model(ridge_reg, X_train, X_test, 'Ridge')
evaluate_model(lasso_reg, X_train, X_test, 'Lasso')
evaluate_model(elasticnet_reg, X_train, X_test, 'ElasticNet OG')

In [32]:
print(result_MAE)

                          model     train      test
0             Linear Regression  786.4448  791.6552
1  Linear Regression Analytical  786.4448  791.6552
2         Linear Regression SGD  795.9364  800.7231
3          Linear Regression GD  784.3144  788.4456
4          Linear Regression L1  786.3229  791.5486
5          Linear Regression L2  782.6626  787.6433
6             ElasticNet custom  784.2193  789.3083
7                         Ridge  786.4344  791.6443
8                         Lasso  785.4790  790.6700
9                 ElasticNet OG  867.7335  868.9930


In [33]:
print(result_RMSE)

                          model      train       test
0             Linear Regression  1252.4692  1498.6112
1  Linear Regression Analytical  1252.4692  1498.6112
2         Linear Regression SGD  1299.8644  1507.6000
3          Linear Regression GD  1263.7868  1445.7239
4          Linear Regression L1  1252.4852  1498.6429
5          Linear Regression L2  1253.6462  1477.0592
6             ElasticNet custom  1252.8178  1487.2136
7                         Ridge  1252.4692  1498.5627
8                         Lasso  1252.6316  1498.6910
9                 ElasticNet OG  1440.9467  1464.4014


In [34]:
print(result_R2)

                          model   train    test
0             Linear Regression  0.5736  0.3888
1  Linear Regression Analytical  0.5736  0.3888
2         Linear Regression SGD  0.5408  0.3814
3          Linear Regression GD  0.5659  0.4312
4          Linear Regression L1  0.5736  0.3888
5          Linear Regression L2  0.5728  0.4062
6             ElasticNet custom  0.5734  0.3980
7                         Ridge  0.5736  0.3888
8                         Lasso  0.5735  0.3887
9                 ElasticNet OG  0.4357  0.4164


## Нормализация признаков (MinMaxScaler)

**Формула:**

$$
X_{\text{scaled}} = \frac{X - X_{\text{min}}}{X_{\text{max}} - X_{\text{min}}}
$$

**Где:**

- $X$ — исходное значение признака  
- $X_{\text{scaled}}$ — нормализованное значение  
- $X_{\text{min}}$ — минимальное значение признака  
- $X_{\text{max}}$ — максимальное значение признака

Нормализация нужна для:
Градиентного спуска (SGD/GD), так как он сильно зависит от масштаба признаков
Методов, основанных на расстоянии (KNN, K-means, SVM)
Регуляризованных моделей (Ridge, Lasso, ElasticNet)

Нормализация не нужна для:
Методов, основанных на деревьях (Decision Tree, Random Forest)
Методов, основанных на нейронных сетях, так как внтури них обычно уже есть нормализация
One-Hot-Encode признаки, так как они уже закодированы в диапазоне [0, 1] 

In [35]:
class MinMaxScalerCustom:
    def __init__(self, feature_range=(0, 1)):
        self.min_ = None
        self.max_ = None
        self.data_min = feature_range[0]
        self.data_max = feature_range[1]


    def fit(self, X):
        X = np.array(X)
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self


    def transform(self, X):
        X = np.array(X)
        scale = self.max_ - self.min_
        scale[scale == 0] = 1  # чтобы не делить на 0, если max == min
        return self.data_min + (X - self.min_) / scale * (self.data_max - self.data_min)
    

    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [36]:
scaler_MM = MinMaxScaler()
X_train_scaled_MM_og = scaler_MM.fit_transform(X_train)
X_test_scaled_MM_og = scaler_MM.transform(X_test)

In [37]:
scaler_MM_custom = MinMaxScalerCustom()
X_train_scaled_MM_custom = scaler_MM_custom.fit_transform(X_train)
X_test_scaled_MM_custom = scaler_MM_custom.transform(X_test)

In [38]:
print(np.allclose(X_test_scaled_MM_custom, X_test_scaled_MM_og))
print(np.allclose(X_train_scaled_MM_custom, X_train_scaled_MM_og))

True
True


## Стандартизация признаков (StandardScaler)

**Формула:**

$$
X_{\text{scaled}} = \frac{X - \mu}{\sigma}
$$

**Где:**

- $X$ — исходное значение признака  
- $X_{\text{scaled}}$ — стандартизованное значение  
- $\mu$ — среднее значение признака  
- $\sigma$ — стандартное отклонение признака

In [39]:
class StandardScaler_custom:
    def __init__(self):
        self.mean_ = None
        self.scale_ = None


    def fit(self, X):
        X = np.array(X)
        self.mean_ = np.mean(X, axis=0)
        self.scale_ = np.std(X, axis=0, ddof=0)
        self.scale_[self.scale_ == 0] = 1.0
        return self


    def transform(self, X):
        return (X - self.mean_) / self.scale_


    def fit_transform(self, X):
        return self.fit(X).transform(X)

In [40]:
standardscaler_og = StandardScaler()
X_train_scaled_std_og = standardscaler_og.fit_transform(X_train)
X_test_scaled_std_og = standardscaler_og.transform(X_test)

In [41]:
standardscaler_custom = StandardScaler_custom()
X_train_scaled_std_custom = standardscaler_custom.fit_transform(X_train)
X_test_scaled_std_custom = standardscaler_custom.transform(X_test)

In [42]:
print(np.array_equal(X_train_scaled_std_og, X_train_scaled_std_custom))
print(np.array_equal(X_test_scaled_std_custom, X_test_scaled_std_og))

True
True


In [43]:
def fit_scaled(model):
    
    result = {
        'MM': {
            'model': type(model)().fit(X_train_scaled_MM_og, y_train),
            'X_train': X_train_scaled_MM_og,
            'X_test': X_test_scaled_MM_og
        },
        'STD': {
            'model': type(model)().fit(X_train_scaled_std_og, y_train),
            'X_train': X_train_scaled_std_og,
            'X_test': X_test_scaled_std_og
        }
    }

    return result

In [44]:
models_list = [lr, ridge_reg, lasso_reg, elasticnet_reg]

for base_model in models_list:
    result_model = fit_scaled(base_model)
    for scaler_tag, data in result_model.items():
        trained_model = data['model']
        X_train = data['X_train']
        X_test = data['X_test']

        model_name = f"{trained_model.__class__.__name__}_{scaler_tag}"

        evaluate_model(trained_model, X_train, X_test, model_name)

In [45]:
df_test_poly = df_test
df_test_poly['interest_level'] = '0'
X_train_3 = df_train[['bathrooms', 'bedrooms', 'interest_level']]

In [46]:
poly = PolynomialFeatures(degree=10, include_bias=False)
X_train_poly = poly.fit_transform(X_train_3)

In [47]:
def evaluate_model_poly(model, X_train, y_train, name):
    y_pred_train = model.predict(X_train)

    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)

    mae_train = f'{mae_train:.4f}'
    rmse_train = f'{rmse_train:.4f}'
    r2_train = f'{r2_train:.4f}'
    
    result_MAE.loc[len(result_MAE)] = [name, mae_train, '-']
    result_RMSE.loc[len(result_RMSE)] = [name, rmse_train, '-']
    result_R2.loc[len(result_R2)] = [name, r2_train, '-']

In [48]:
X_train_poly_scaled = standardscaler_og.fit_transform(X_train_poly)

linreg_poly = LinearRegressionAnalytical()
linreg_poly.fit(X_train_poly_scaled, y_train)

evaluate_model_poly(linreg_poly, X_train_poly_scaled, y_train, 'Linear Regression Poly')

lasso_poly = LinearRegressionRegularized(penalty='l1')
lasso_poly.fit(X_train_poly_scaled, y_train)

evaluate_model_poly(lasso_poly, X_train_poly_scaled, y_train, 'Lasso Poly')

ridge_poly = LinearRegressionRegularized(penalty='l2')
ridge_poly.fit(X_train_poly_scaled, y_train)

evaluate_model_poly(ridge_poly, X_train_poly_scaled, y_train, 'Ridge Poly')

elasticnet_poly = LinearRegressionRegularized(penalty='elasticnet')
elasticnet_poly.fit(X_train_poly_scaled, y_train)

evaluate_model_poly(elasticnet_poly, X_train_poly_scaled, y_train, 'ElasticNet Poly')

Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:39<00:00, 255.60it/s]


In [49]:
print(result_MAE)

                           model      train       test
0              Linear Regression   786.4448   791.6552
1   Linear Regression Analytical   786.4448   791.6552
2          Linear Regression SGD   795.9364   800.7231
3           Linear Regression GD   784.3144   788.4456
4           Linear Regression L1   786.3229   791.5486
5           Linear Regression L2   782.6626   787.6433
6              ElasticNet custom   784.2193   789.3083
7                          Ridge   786.4344   791.6443
8                          Lasso   785.4790   790.6700
9                  ElasticNet OG   867.7335   868.9930
10           LinearRegression_MM   786.4448   791.6552
11          LinearRegression_STD   786.4448   791.6552
12                      Ridge_MM   786.1482   791.3028
13                     Ridge_STD   786.4429   791.6531
14                      Lasso_MM   784.8414   789.8960
15                     Lasso_STD   786.0376   791.2414
16                 ElasticNet_MM  1154.2482  1150.9646
17        

In [50]:
print(result_RMSE)

                           model      train       test
0              Linear Regression  1252.4692  1498.6112
1   Linear Regression Analytical  1252.4692  1498.6112
2          Linear Regression SGD  1299.8644  1507.6000
3           Linear Regression GD  1263.7868  1445.7239
4           Linear Regression L1  1252.4852  1498.6429
5           Linear Regression L2  1253.6462  1477.0592
6              ElasticNet custom  1252.8178  1487.2136
7                          Ridge  1252.4692  1498.5627
8                          Lasso  1252.6316  1498.6910
9                  ElasticNet OG  1440.9467  1464.4014
10           LinearRegression_MM  1252.4692  1498.6112
11          LinearRegression_STD  1252.4692  1498.6112
12                      Ridge_MM  1252.5076  1493.0061
13                     Ridge_STD  1252.4692  1498.6022
14                      Lasso_MM  1252.7964  1487.9026
15                     Lasso_STD  1252.4933  1498.6865
16                 ElasticNet_MM  1794.3129  1793.6613
17        

In [51]:
print(result_R2)

                           model   train    test
0              Linear Regression  0.5736  0.3888
1   Linear Regression Analytical  0.5736  0.3888
2          Linear Regression SGD  0.5408  0.3814
3           Linear Regression GD  0.5659  0.4312
4           Linear Regression L1  0.5736  0.3888
5           Linear Regression L2  0.5728  0.4062
6              ElasticNet custom  0.5734  0.3980
7                          Ridge  0.5736  0.3888
8                          Lasso  0.5735  0.3887
9                  ElasticNet OG  0.4357  0.4164
10           LinearRegression_MM  0.5736  0.3888
11          LinearRegression_STD  0.5736  0.3888
12                      Ridge_MM  0.5736  0.3933
13                     Ridge_STD  0.5736  0.3888
14                      Lasso_MM  0.5734  0.3975
15                     Lasso_STD  0.5736  0.3887
16                 ElasticNet_MM  0.1249  0.1244
17                ElasticNet_STD  0.5364  0.4469
18        Linear Regression Poly  0.6069       -
19                  

лучшей моделю является Linear Regression GD

In [52]:
alphas = [100, 200, 300, 400, 500, 1000]

for alpha in alphas:
    lasso_poly = LinearRegressionRegularized(penalty='l1', alpha=alpha)
    lasso_poly.fit(X_train_poly_scaled, y_train)

    y_pred_train = lasso_poly.predict(X_train_poly_scaled)

    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)

    print(f'{mae_train:.4f}', alpha)
    print(f'{rmse_train:.4f}')
    print(f'{r2_train:.4f}')

Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:38<00:00, 257.04it/s]


784.5282 100
1225.1198
0.5920


Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:39<00:00, 254.12it/s]


784.5292 200
1225.1214
0.5920


Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:39<00:00, 255.03it/s]


...

784.5311 400
1225.1247
0.5920


Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:40<00:00, 243.93it/s]


784.5321 500
1225.1264
0.5920


Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:40<00:00, 244.33it/s]

784.5374 1000
1225.1351
0.5920


In [53]:
for alpha in alphas:
    lasso_poly = LinearRegressionRegularized(penalty='l2', alpha=alpha)
    lasso_poly.fit(X_train_poly_scaled, y_train)

    y_pred_train = lasso_poly.predict(X_train_poly_scaled)

    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)

    print(f'{mae_train:.4f}', alpha)
    print(f'{rmse_train:.4f}')
    print(f'{r2_train:.4f}')

Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:44<00:00, 227.06it/s]


784.8524 100
1225.6392
0.5917


Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:51<00:00, 194.59it/s]


785.2245 200
1226.1813
0.5913


Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:40<00:00, 248.55it/s]


...

785.9768 400
1227.2777
0.5906


Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:39<00:00, 252.76it/s]


786.3401 500
1227.8156
0.5903


Training: 100%|██████████████████████████████████████████████████████████| 10000/10000 [00:39<00:00, 252.70it/s]

788.2691 1000
1230.2647
0.5886


результаты ухудшаются с увеличением alpha

In [54]:
mean_price_train = y_train.mean()
median_price_train = y_train.median()

mean_price_test = y_true.mean()
median_price_test = y_true.median()

result_MAE.loc[len(result_MAE)] = [
    'Naive mean',
    mean_absolute_error(y_train, np.full_like(y_train, mean_price_train)),
    mean_absolute_error(y_true, np.full_like(y_true, mean_price_test))
]
result_RMSE.loc[len(result_RMSE)] = [
    'Naive mean',
    np.sqrt(mean_squared_error(y_train, np.full_like(y_train, mean_price_train))),
    np.sqrt(mean_squared_error(y_true, np.full_like(y_true, mean_price_test)))
]
result_R2.loc[len(result_R2)] = [
    'Naive mean',
    r2_score(y_train, np.full_like(y_train, mean_price_train)),
    r2_score(y_true, np.full_like(y_true, mean_price_test))
]

result_MAE.loc[len(result_MAE)] = [
    'Naive median',
    mean_absolute_error(y_train, np.full_like(y_train, median_price_train)),
    mean_absolute_error(y_true, np.full_like(y_true, median_price_test))
]
result_RMSE.loc[len(result_RMSE)] = [
    'Naive median',
    np.sqrt(mean_squared_error(y_train, np.full_like(y_train, median_price_train))),
    np.sqrt(mean_squared_error(y_true, np.full_like(y_true, median_price_test)))
]
result_R2.loc[len(result_R2)] = [
    'Naive median',
    r2_score(y_train, np.full_like(y_train, median_price_train)),
    r2_score(y_true, np.full_like(y_true, median_price_test))
]

In [55]:
print(result_MAE)

                           model        train         test
0              Linear Regression     786.4448     791.6552
1   Linear Regression Analytical     786.4448     791.6552
2          Linear Regression SGD     795.9364     800.7231
3           Linear Regression GD     784.3144     788.4456
4           Linear Regression L1     786.3229     791.5486
5           Linear Regression L2     782.6626     787.6433
6              ElasticNet custom     784.2193     789.3083
7                          Ridge     786.4344     791.6443
8                          Lasso     785.4790     790.6700
9                  ElasticNet OG     867.7335     868.9930
10           LinearRegression_MM     786.4448     791.6552
11          LinearRegression_STD     786.4448     791.6552
12                      Ridge_MM     786.1482     791.3028
13                     Ridge_STD     786.4429     791.6531
14                      Lasso_MM     784.8414     789.8960
15                     Lasso_STD     786.0376     791.24

In [56]:
print(result_RMSE)

                           model        train         test
0              Linear Regression    1252.4692    1498.6112
1   Linear Regression Analytical    1252.4692    1498.6112
2          Linear Regression SGD    1299.8644    1507.6000
3           Linear Regression GD    1263.7868    1445.7239
4           Linear Regression L1    1252.4852    1498.6429
5           Linear Regression L2    1253.6462    1477.0592
6              ElasticNet custom    1252.8178    1487.2136
7                          Ridge    1252.4692    1498.5627
8                          Lasso    1252.6316    1498.6910
9                  ElasticNet OG    1440.9467    1464.4014
10           LinearRegression_MM    1252.4692    1498.6112
11          LinearRegression_STD    1252.4692    1498.6112
12                      Ridge_MM    1252.5076    1493.0061
13                     Ridge_STD    1252.4692    1498.6022
14                      Lasso_MM    1252.7964    1487.9026
15                     Lasso_STD    1252.4933    1498.68

In [57]:
print(result_R2)

                           model     train      test
0              Linear Regression    0.5736    0.3888
1   Linear Regression Analytical    0.5736    0.3888
2          Linear Regression SGD    0.5408    0.3814
3           Linear Regression GD    0.5659    0.4312
4           Linear Regression L1    0.5736    0.3888
5           Linear Regression L2    0.5728    0.4062
6              ElasticNet custom    0.5734    0.3980
7                          Ridge    0.5736    0.3888
8                          Lasso    0.5735    0.3887
9                  ElasticNet OG    0.4357    0.4164
10           LinearRegression_MM    0.5736    0.3888
11          LinearRegression_STD    0.5736    0.3888
12                      Ridge_MM    0.5736    0.3933
13                     Ridge_STD    0.5736    0.3888
14                      Lasso_MM    0.5734    0.3975
15                     Lasso_STD    0.5736    0.3887
16                 ElasticNet_MM    0.1249    0.1244
17                ElasticNet_STD    0.5364    

лучшая модель по качеству - LinearRegression GD, почти все модели ведут себя стабильно, показывая небольшую разницу между результатами на train и test